# PROMISE Multi-Project AST Extraction

This notebook runs and inspects the generalized AST extraction pipeline implemented in `scripts/extract_promise_ast.py`.

The script is the single source of truth. The notebook only wraps execution and validates the generated graph/tensor outputs.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SCRIPT_PATH = REPO_ROOT / 'scripts' / 'extract_promise_ast.py'
INPUT_CSV = REPO_ROOT / 'outputs' / 'promise' / 'promise_preprocessed_standard.csv'
AST_DIR = REPO_ROOT / 'outputs' / 'promise' / 'ast'
SUMMARY_JSON = AST_DIR / 'ast_summary.json'
GRAPH_INDEX = AST_DIR / 'graph_index.csv'
REPORT_MD = AST_DIR / 'ast_report.md'

print('Repository:', REPO_ROOT)
print('AST extractor:', SCRIPT_PATH)
print('Input CSV:', INPUT_CSV)
print('AST output directory:', AST_DIR)

## 1. Design

The extractor creates one AST graph for every mapped Java class in the combined preprocessed PROMISE dataset.

Each graph stores:

- structural node features: `x = [depth, out_degree, has_identifier]`
- exact AST node type ids: `node_type_id`
- AST parent-child edges: `edge_index`

During GNN training, `node_type_id` should be passed through a trainable embedding layer and concatenated with the structural features.

In [ ]:
pre_df = pd.read_csv(INPUT_CSV)
print('combined preprocessing shape:', pre_df.shape)
print('datasets:', sorted(pre_df['dataset_name'].unique()))
print('mapped rows:', int(pre_df['source_path'].notna().sum()))
pre_df.head()

## 2. Run AST Extraction

Terminal equivalent:

```bash
python scripts/extract_promise_ast.py
```

To run one dataset only:

```bash
python scripts/extract_promise_ast.py --dataset-name ant-1.6
```

In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPT_PATH)],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)

## 3. Extraction Summary

In [ ]:
summary = json.loads(SUMMARY_JSON.read_text())
summary_view = {
    key: summary[key]
    for key in [
        'input_rows',
        'requested_mapped_samples',
        'graphs_generated',
        'parse_failures',
        'fallback_graphs',
        'total_nodes',
        'total_edges',
        'avg_nodes_per_graph',
        'avg_edges_per_graph',
        'structural_feature_dim',
        'node_type_embedding_dim',
        'model_node_feature_dim',
        'node_type_vocab_size',
    ]
}
summary_view

## 4. Per-Dataset Results

In [ ]:
pd.DataFrame(summary['datasets'])

## 5. Graph Index

In [ ]:
graph_index = pd.read_csv(GRAPH_INDEX)
print('graph_index shape:', graph_index.shape)
graph_index.head()

## 6. Tensor Shape Check For One Graph

In [ ]:
example = graph_index.iloc[0]
x = np.load(example['x_npy'])
node_type_id = np.load(example['node_type_id_npy'])
edge_index = np.load(example['edge_index_npy'])

print('graph_id:', example['graph_id'])
print('x:', x.shape)
print('node_type_id:', node_type_id.shape)
print('edge_index:', edge_index.shape)
print('finite x:', np.isfinite(x).all())
print('node_type_id range:', int(node_type_id.min()), int(node_type_id.max()))

## 7. Validation Checks

In [ ]:
checks = {
    'graphs_match_requested_minus_failures': summary['graphs_generated'] == summary['requested_mapped_samples'] - summary['parse_failures'],
    'graph_index_rows_match_summary': len(graph_index) == summary['graphs_generated'],
    'structural_feature_dim_is_3': summary['structural_feature_dim'] == 3,
    'model_feature_dim_is_35': summary['model_node_feature_dim'] == 35,
    'example_x_rows_match_node_type_ids': x.shape[0] == node_type_id.shape[0],
    'example_edge_index_has_two_rows': edge_index.shape[0] == 2,
    'example_x_has_three_features': x.shape[1] == 3,
}
checks

## 8. Report Preview

In [ ]:
print(REPORT_MD.read_text()[:5000])